# Lab 21-12 · The Modem in Numbers

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import modem as m

## Part 1 · Peak rates

*Predict* the rate of two 100 MHz carriers (4 layers, 256-QAM) plus a 20 MHz carrier (2 layers, 64-QAM).

In [2]:
three = 2 * m.nr_rate_mbps(4, 8, 273, 1, 0.14) + m.nr_rate_mbps(2, 6, 106, 0, 0.14)
five = 3 * m.nr_rate_mbps(4, 10, 273, 1, 0.14) + 2 * m.nr_rate_mbps(4, 10, 106, 1, 0.14)
print(f"three carriers {three:.1f} Mb/s; five carriers {five:.0f} Mb/s; ratio {five/three:.2f}")
print(f"L2 buffer for 10 + 3.5 Gb/s at 40 ms: {m.l2_buffer_bytes(10e9, 3.5e9, 0.04)/1e6:.1f} MB")

three carriers 4844.1 Mb/s; five carriers 11032 Mb/s; ratio 2.28
L2 buffer for 10 + 3.5 Gb/s at 40 ms: 67.5 MB


## Part 2 · Two-way ranging

*Predict* the rms and worst single-sided error for clocks within ±20 ppm and a 1 ms reply.

In [3]:
rng = np.random.default_rng(5)
tau = 10 / m.C0
ss, ds = [], []
for _ in range(1000):
    ea, eb = rng.uniform(-20e-6, 20e-6, 2)
    s, d = m.twr_simulate(tau, 1e-3, 1e-3, ea, eb)
    ss.append((s - tau) * m.C0); ds.append((d - tau) * m.C0)
ss, ds = np.array(ss), np.array(ds)
print(f"single-sided rms {np.sqrt(np.mean(ss**2)):.2f} m, worst {np.max(np.abs(ss)):.2f} m; double-sided worst {1e3*np.max(np.abs(ds)):.3f} mm")
plt.figure(); plt.hist(ss, 40); plt.xlabel("single-sided range error (m)"); plt.show()

single-sided rms 2.45 m, worst 5.95 m; double-sided worst 0.198 mm


## Part 3 · Phase-based ranging

*Predict* what a 72-tone sweep reports for a 12 m path with a half-amplitude reflection at 20 m.

In [4]:
fr = 2402e6 + np.arange(72) * 1e6
h = np.exp(-1j * 4 * np.pi * fr * 12 / m.C0) + 0.5 * np.exp(-1j * 4 * np.pi * fr * 20 / m.C0)
print(f"slope {m.pbr_slope_distance(fr, np.angle(h)):.2f} m, IFFT {m.pbr_ifft_distance(fr, h):.2f} m")
n = 72 * 16
prof = np.abs(np.fft.ifft(h, n))
dist = np.arange(n) / (n * 1e6) * m.C0 / 2
plt.figure(); plt.plot(dist, prof); plt.xlim(0, 40); plt.xlabel("distance (m)"); plt.ylabel("|IFFT|"); plt.show()

slope 11.94 m, IFFT 11.97 m


## Part 4 · Ionosphere-free noise

*Predict* the noise of the L1/L5 combination with 1 m of independent noise on each pseudorange.

In [5]:
a, b = m.iono_free_coeffs(1575.42e6, 1176.45e6)
n1, n5 = rng.normal(0, 1, (2, 200000))
print(f"coefficients {a:.3f}, {b:.3f}; predicted {math.hypot(a, b):.3f} m; simulated {np.std(a*n1 + b*n5):.3f} m")

coefficients 2.261, -1.261; predicted 2.588 m; simulated 2.591 m


## Part 5 · Doppler over a pass

*Predict* the Doppler at 10°, 30° and 60° elevation for a 600 km satellite at 2 GHz.

In [6]:
for el in (10, 30, 60):
    print(f"{el:2d} deg: {m.leo_doppler_hz(2e9, 600e3, el)/1e3:.1f} kHz")
els = np.linspace(5, 90, 200)
plt.figure(); plt.plot(els, [m.leo_doppler_hz(2e9, 600e3, e)/1e3 for e in els])
plt.xlabel("elevation (deg), approaching"); plt.ylabel("Doppler (kHz)"); plt.show()

10 deg: 45.4 kHz
30 deg: 39.9 kHz
60 deg: 23.1 kHz


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.